# Bước 4: Data Analysis with Python
## Thống Kê Mô Tả Chuyên Sâu, Khám Phá Dữ Liệu (EDA) & Kiểm Định Giả Thuyết Thống Kê Suy Diễn
### Phân Tích Hành Vi Mua Hàng Của Khách Hàng (Tập Dữ Liệu Sạch: 432.256 Bản Ghi)
---

## 4.1 Nạp Thư Viện & Dữ Liệu Sạch (432.256 Dòng)

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv('../Data/customerData_clean.csv')
print(f"Kích thước dữ liệu sạch: {df.shape[0]:,} dòng × {df.shape[1]} cột")

Kích thước dữ liệu sạch: 432,256 dòng × 19 cột


## 4.2 Thống Kê Mô Tả Nâng Cao (Descriptive Statistics: Skewness & Kurtosis)

**Câu hỏi nghiên cứu:** Các biến số định lượng phân phối như thế nào? Có bị lệch (Skewed) hay có đuôi dày (Kurtosis) không?
**Cách làm:** Tính toán đầy đủ Mean, Median, Std, Min, Max, Độ lệch (Skewness) và Độ nhọn (Kurtosis).


In [2]:
num_cols = ['Age', 'AnnualIncome', 'NumberOfPurchases', 'TimeSpentOnWebsite', 
            'CustomerTenureYears', 'LastPurchaseDaysAgo', 'DiscountsAvailed', 'SessionCount']

desc_stats = []
for c in num_cols:
    s = df[c]
    desc_stats.append({
        'Biến số': c,
        'Trung bình (Mean)': np.round(s.mean(), 2),
        'Trung vị (Median)': np.round(s.median(), 2),
        'Độ lệch chuẩn (Std)': np.round(s.std(), 2),
        'Nhỏ nhất (Min)': np.round(s.min(), 2),
        'Lớn nhất (Max)': np.round(s.max(), 2),
        'Độ lệch (Skewness)': np.round(stats.skew(s), 3),
        'Độ nhọn (Kurtosis)': np.round(stats.kurtosis(s), 3)
    })

desc_df = pd.DataFrame(desc_stats)
display(desc_df)


,Biến số,Trung bình (Mean),Trung vị (Median),Độ lệch chuẩn (Std),Nhỏ nhất (Min),Lớn nhất (Max),Độ lệch (Skewness),Độ nhọn (Kurtosis)
0,Age,44.15,44.00,15.65,15.00,81.00,-0.010,-1.142
1,AnnualIncome,85018.02,83686.29,39578.42,11966.39,204178.29,0.178,-0.923
2,NumberOfPurchases,11.59,12.00,5.86,0.00,27.00,-0.072,-1.071
3,TimeSpentOnWebsite,31.15,31.29,17.23,0.00,78.36,0.019,-1.048
4,CustomerTenureYears,2.30,1.64,2.14,0.00,15.35,1.851,4.520
5,LastPurchaseDaysAgo,57.58,30.00,53.24,0.00,189.00,0.854,-0.657
6,DiscountsAvailed,3.15,3.00,1.88,0.00,10.00,0.057,-0.812
7,SessionCount,2.35,2.00,1.48,1.00,12.00,1.223,1.484


> **Nhận xét thống kê mô tả:**
> - Các biến `Age` (Skewness = **-0.010**) và `TimeSpentOnWebsite` (Skewness = **0.019**) có độ lệch rất gần 0 (nằm trong khoảng `-0.05 đến 0.05`), cùng với `DiscountsAvailed` (**0.057**) và `NumberOfPurchases` (**-0.072**), biểu thị phân phối gần như đối xứng hoàn hảo.
> - Biến `AnnualIncome` có độ lệch phải nhẹ (Skewness = **0.178**, nhưng vẫn rất thấp so với ngưỡng cảnh báo lệch nặng $|\text{Skew}| > 1$), với giá trị trung bình (**85,018.02 USD**) và trung vị (**83,686.29 USD**) khá sát nhau.
> - Riêng `CustomerTenureYears` (**1.851**) và `SessionCount` (**1.223**) có độ lệch phải rõ rệt (đa số khách hàng có thâm niên ngắn và số phiên ít). Nhìn chung, sau khi áp dụng bộ 7 quy tắc làm sạch DAMA-DMBOK (loại bỏ 67.744 dòng lỗi), dữ liệu có độ ổn định cao và không bị méo lệch bởi outlier cực đoan.


## 4.3 Ma trận Tương quan Tuyến tính Pearson (Correlation Analysis)

**Câu hỏi nghiên cứu:** Giữa các biến định lượng và biến mục tiêu `PurchaseStatus`, những cặp biến nào có tương quan mạnh nhất?


In [3]:
corr_matrix = df[num_cols + ['CustomerSatisfaction', 'PurchaseStatus']].corr()
purchase_corr = corr_matrix['PurchaseStatus'].sort_values(ascending=False).to_frame(name='Hệ số tương quan với PurchaseStatus')
print("HỆ SỐ TƯƠNG QUAN VỚI BIẾN KẾT QUẢ (PURCHASE STATUS):")
display(purchase_corr)


HỆ SỐ TƯƠNG QUAN VỚI BIẾN KẾT QUẢ (PURCHASE STATUS):


,Hệ số tương quan với PurchaseStatus
PurchaseStatus,1.000000
CustomerSatisfaction,0.132825
AnnualIncome,0.091954
NumberOfPurchases,0.067501
TimeSpentOnWebsite,0.065100
DiscountsAvailed,0.054506
SessionCount,0.001639
CustomerTenureYears,-0.040499
Age,-0.066688
LastPurchaseDaysAgo,-0.691567


> **Nhận xét ma trận tương quan (Khớp chuẩn xác với Báo cáo 2):**
> - **Tương quan âm mạnh nhất — Biến chi phối hàng đầu:**
>   * `LastPurchaseDaysAgo` ($r = -0.692$): Hệ số tương quan âm rất mạnh với `PurchaseStatus`. Khách hàng có số ngày kể từ lần mua cuối càng lâu thì xác suất chốt đơn ở phiên này càng sụt giảm nghiêm trọng (hiệu ứng Recency cliff). 
> - **Top các tương quan dương:**
>   1. `CustomerSatisfaction` ($r = +0.133$): Điểm đánh giá trải nghiệm tỷ lệ thuận với tỷ lệ mua sắm. Tuy nhiên, đây là biến phát sinh sau giao dịch (T2), do đó cần loại bỏ ở Bước 6 để chống rò rỉ dữ liệu (Look-ahead Leakage).
>   2. `AnnualIncome` ($r = +0.092$): Khách hàng có thu nhập cao có sức mua ổn định và tỷ lệ chốt đơn cao hơn.
>   3. `NumberOfPurchases` ($r = +0.068$) & `TimeSpentOnWebsite` ($r = +0.065$): Mức độ gắn kết và thời lượng tương tác tích cực trên nền tảng.
>   4. `DiscountsAvailed` ($r = +0.055$): Số lượng mã giảm giá hỗ trợ kích cầu nhẹ.



## 4.4 Kiểm Định Giả Thuyết Thống Kê (Two-Sample Independent T-Test)

**Câu hỏi nghiên cứu:** Giá trị trung bình của 8 biến định lượng (`AnnualIncome`, `TimeSpentOnWebsite`, `NumberOfPurchases`, `Age`, `DiscountsAvailed`, `SessionCount`, `LastPurchaseDaysAgo`, `CustomerTenureYears`) ở nhóm Mua hàng (`PurchaseStatus = 1`) có sự khác biệt mang ý nghĩa thống kê so với nhóm Không mua (`PurchaseStatus = 0`) hay không?
* **Giả thuyết kiểm định (cho từng biến định lượng):**
  * $H_0$: $\mu_1 = \mu_0$ (Không có sự khác biệt về giá trị trung bình giữa nhóm mua và không mua).
  * $H_1$: $\mu_1 \neq \mu_0$ (Có sự khác biệt có ý nghĩa thống kê giữa 2 nhóm).
  * Mức ý nghĩa: $\alpha = 0.05$.


In [4]:
group_1 = df[df['PurchaseStatus'] == 1]
group_0 = df[df['PurchaseStatus'] == 0]

test_vars = ['AnnualIncome', 'TimeSpentOnWebsite', 'NumberOfPurchases', 'Age', 'DiscountsAvailed', 'SessionCount', 'LastPurchaseDaysAgo', 'CustomerTenureYears']
ttest_results = []

for var in test_vars:
    t_stat, p_val = stats.ttest_ind(group_1[var], group_0[var], equal_var=False)
    diff = group_1[var].mean() - group_0[var].mean()
    ttest_results.append({
        'Biến kiểm định': var,
        'Mean (Mua = 1)': np.round(group_1[var].mean(), 2),
        'Mean (Không mua = 0)': np.round(group_0[var].mean(), 2),
        'Chênh lệch (Diff)': np.round(diff, 2),
        'T-Statistic': np.round(t_stat, 2),
        'P-Value': f"{p_val:.2e}",
        'Ý nghĩa thống kê (p < 0.05)': 'Bác bỏ H0 (Có ý nghĩa)' if p_val < 0.05 else 'Chấp nhận H0'
    })

ttest_df = pd.DataFrame(ttest_results)
display(ttest_df)


,Biến kiểm định,Mean (Mua = 1),Mean (Không mua = 0),Chênh lệch (Diff),T-Statistic,P-Value,Ý nghĩa thống kê (p < 0.05)
0,AnnualIncome,89178.92,81834.77,7344.14,60.64,0.00e+00,Bác bỏ H0 (Có ý nghĩa)
1,TimeSpentOnWebsite,32.43,30.17,2.26,42.90,0.00e+00,Bác bỏ H0 (Có ý nghĩa)
2,NumberOfPurchases,12.04,11.25,0.80,44.54,0.00e+00,Bác bỏ H0 (Có ý nghĩa)
3,Age,42.96,45.06,-2.11,-43.95,0.00e+00,Bác bỏ H0 (Có ý nghĩa)
4,DiscountsAvailed,3.27,3.06,0.21,35.91,6.97e-282,Bác bỏ H0 (Có ý nghĩa)
5,SessionCount,2.35,2.35,0.00,1.08,2.81e-01,Chấp nhận H0
6,LastPurchaseDaysAgo,15.48,89.79,-74.31,-712.70,0.00e+00,Bác bỏ H0 (Có ý nghĩa)
7,CustomerTenureYears,2.20,2.38,-0.17,-26.63,4.29e-156,Bác bỏ H0 (Có ý nghĩa)


> **Kết luận kiểm định T-Test:**
> * **7/8 biến có ý nghĩa ($p < 0.001$):** Nhóm mua nổi bật nhất ở **độ mới giao dịch gần** (15.5 vs 89.8 ngày) và **thu nhập cao hơn** (+$7,344 USD).
> * **Loại bỏ `SessionCount` ($p = 0.281 > 0.05$):** Tần suất vào web giữa 2 nhóm bằng nhau (2.35 phiên), không ảnh hưởng đến quyết định mua nên bị loại

## 4.5 Phân Tích Phương Sai Một Yếu Tố (One-Way ANOVA)

**Câu hỏi nghiên cứu:** Thời gian trên website (`TimeSpentOnWebsite`) có sự khác biệt có ý nghĩa thống kê giữa 3 phân khúc khách hàng (`Regular`, `Premium`, `VIP`) hay không?
* **Giả thuyết kiểm định:**
  * $H_0$: $\mu_1 = \mu_2 = \mu_3$.
  * $H_1$: Có ít nhất một cặp phân khúc có thời gian trung bình khác nhau.


In [ ]:
seg_groups = [group['TimeSpentOnWebsite'].values for _, group in df.groupby('CustomerSegment')]
f_stat, p_val = stats.f_oneway(*seg_groups)

print("KẾT QUẢ KIỂM ĐỊNH ONE-WAY ANOVA (TimeSpentOnWebsite theo CustomerSegment):")
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value    : {p_val:.4e}")


if p_val < 0.05:
    print(" KẾT LUẬN: Bác bỏ H0 — Thời gian truy cập website giữa các phân khúc khách hàng có sự khác biệt rõ rệt!")
else:
    print(" KẾT LUẬN: Không đủ bằng chứng bác bỏ H0.")


> **Kết luận kiểm định ANOVA:**
> - Với $F = 0.0244$ và $p = 0.9759 > 0.05$, ta **Chấp nhận giả thuyết $H_0$** (Không có sự khác biệt có ý nghĩa thống kê về thời gian duyệt web giữa các nhóm khách hàng).



## 4.6 Kiểm Định Chi-Square ($\chi^2$) về Tính Độc Lập & Hệ Số Cramér's V

**Câu hỏi nghiên cứu:** Quyết định mua hàng (`PurchaseStatus`) có độc lập với toàn bộ 7 biến phân loại (`LoyaltyProgram`, `CustomerSegment`, `ProductCategory`, `Gender`, `Region`, `PreferredDevice`, `ReferralSource`) hay không? 
* **Phương pháp:** Kiểm định $\chi^2$ (Chi-square test of independence) và đo độ mạnh liên kết bằng hệ số **Cramér's V**:
  $$V = \sqrt{\frac{\chi^2}{n \times \min(r-1, c-1)}}$$


In [ ]:
def cramers_v(contingency_table):
    chi2 = stats.chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    r, c = contingency_table.shape
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))

# Kiểm định Chi-Square & Đo lường độ mạnh liên kết Cramér's V cho toàn bộ 7 biến phân loại
cat_test_cols = ['LoyaltyProgram', 'CustomerSegment', 'PreferredDevice', 'ReferralSource', 'Region', 'ProductCategory', 'Gender']
chi2_results = []

for col in cat_test_cols:
    contingency = pd.crosstab(df[col], df['PurchaseStatus'])
    chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
    v = cramers_v(contingency)
    chi2_results.append({
        'Biến phân loại': col,
        'Bậc tự do (dof)': dof,
        'Chi-Square (χ²)': np.round(chi2, 2),
        'P-Value': f"{p_val:.2e}",
        'Cramer_V (Độ mạnh)': np.round(v, 4),
        'Mối liên hệ': 'Có ý nghĩa (Phụ thuộc)' if p_val < 0.05 else 'Độc lập'
    })


chi2_df = pd.DataFrame(chi2_results).sort_values(by='Cramer_V (Độ mạnh)', ascending=False).reset_index(drop=True)
display(chi2_df)

,Biến phân loại,Bậc tự do (dof),Chi-Square (χ²),P-Value,Cramer_V (Độ mạnh),Mối liên hệ
0,LoyaltyProgram,1,3125.90,0.00e+00,0.0850,Có ý nghĩa (Phụ thuộc)
1,CustomerSegment,2,1422.03,1.62e-309,0.0574,Có ý nghĩa (Phụ thuộc)
5,ProductCategory,4,89.29,1.86e-18,0.0144,Có ý nghĩa (Phụ thuộc)
6,Gender,1,46.99,7.14e-12,0.0104,Có ý nghĩa (Phụ thuộc)
4,Region,3,15.75,1.28e-03,0.0060,Có ý nghĩa (Phụ thuộc)
2,PreferredDevice,2,13.80,1.01e-03,0.0057,Có ý nghĩa (Phụ thuộc)
3,ReferralSource,4,9.94,4.15e-02,0.0048,Có ý nghĩa (Phụ thuộc)


> **Kết luận kiểm định Chi-Square & Cramér's V (Đủ 7 biến phân loại):**
>
> - Tất cả 7 biến phân loại đều có $p < 0.05$, bác bỏ giả thuyết độc lập ($H_0$).
>Về độ mạnh thực tế (Theo đúng thang đo Cramér's V):
Tất cả 7 biến đều có 𝑉 ≤ 0.1 ⟹ Mức độ liên hệ thực tế đều ở mức "Rất yếu hoặc không đáng kể"!
Hoàn toàn không có biến phân loại nào đạt mức liên hệ Yếu (>0.1), Trung bình (>0.3) hay Mạnh (>0.5).

## 4.7 Phân Tích Phân Khúc RFM (Recency & Frequency Matrix)

**Câu hỏi nghiên cứu:** Khách hàng có độ mới giao dịch gần (Recency thấp) và tần suất mua sắm cao (Frequency cao) có xác suất mua sắm đạt mức bao nhiêu %?


In [7]:
df['RecencyGroup'] = pd.qcut(df['LastPurchaseDaysAgo'], q=3, labels=['Active (Gần)', 'Warm (Vừa)', 'Cold (Xa)'])
df['FreqGroup'] = pd.qcut(df['NumberOfPurchases'], q=3, labels=['Low Freq', 'Med Freq', 'High Freq'])

rfm_pivot = df.pivot_table(
    index='RecencyGroup',
    columns='FreqGroup',
    values='PurchaseStatus',
    aggfunc=['count', lambda x: np.round(x.mean() * 100, 2)]
)
rfm_pivot.columns = ['Count: Low', 'Count: Med', 'Count: High', 'CVR(%): Low', 'CVR(%): Med', 'CVR(%): High']
print("MA TRẬN CHUYỂN ĐỔI THEO KHUNG RFM (RECENCY & FREQUENCY):")
display(rfm_pivot)


MA TRẬN CHUYỂN ĐỔI THEO KHUNG RFM (RECENCY & FREQUENCY):


,Count: Low,Count: Med,Count: High,CVR(%): Low,CVR(%): Med,CVR(%): High
RecencyGroup,,,,,,
Active (Gần),55175,46899,43508,85.13,90.45,93.85
Warm (Vừa),53809,46234,43024,32.84,41.04,47.59
Cold (Xa),53977,46485,43145,0.00,0.01,0.01


> **Kết luận phân tích RFM:**
> - Nhóm khách hàng **Active + High Freq** (mới mua gần đây và mua nhiều lần) sở hữu tỷ lệ chuyển đổi cao nhất nền tảng.
> - Nhóm **Cold + Low Freq** có tỷ lệ chuyển đổi thấp nhất, cần áp dụng các chiến dịch Win-back (kích hoạt lại) với ưu đãi đặc thù.
